# 1. Definição de escopo e objetivo

## 1.1. Introdução: o panorama jurídico das execuções fiscais

A cobrança de créditos públicos inscritos em dívida ativa é realizada por meio do processo de execução fiscal, conforme previsão da Lei n.º 6.830.

Porém, a demora inerente ao processo judicial somada com a utilização massificada e pouco seletiva do instituto terminou por resultar em um quadro de congestionamento do Poder Judiciário brasileiro. Nesse sentido, o estudo Justiça em Números, publicado anualmente pelo Conselho Nacional de Justiça, vem em diversas de suas edições apresentando as estatísticas que evidenciam que os processos de execução relacionam-se com esse relevante gargalo.

Com o propósito de contornar esse problema, o Supremo Tribunal Federal, ao julgar o Tema nº 1.184, estabeleceu importantes premissas para racionalizar a utilização da execução fiscal, reconhecendo a possibilidade de extinção de execuções de baixo valor e a necessidade de tentativa de cobrança ou regularização administrativa antes de ser utilizada a via judicial.

Acompanhando as conclusões do Supremo Tribunal Federal, o Conselho Nacional de Justiça editou, em 2024, a Resolução CNJ n.º 547, reproduzindo as mencionadas condicionantes em um ato normativo que orienta a atuação do Judiciário nacional.

Passados 2 anos de aplicação da normativa, mostra-se importante investigar se o propósito de redução do número de execuções fiscais foi atingido e como essa diminuição ocorreu nos diferentes órgãos do Poder Judiciário.

##1.2. Escopo do MVP e análise pretendida

Por meio do presente MVP, pretende-se investigar, de forma específica, os resultados da Resolução CNJ n.º 547 na Justiça Estadual de São Paulo.

A partir da coleta dos dados e de sua análise, objetiva-se responder os seguintes questionamentos:

1) Após a edição da Resolução CNJ n.º 547, houve redução das execuções fiscais na Justiça Estadual de São Paulo?

2) Qual o perfil de distribuição das execuções fiscais entre os órgãos do Tribunal de Justiça do Estado de São Paulo?

3) Eventual redução decorrente da aplicação da Resolução CNJ n.º 547 ocorreu de maneira uniforme entre os diversos órgãos?


##1.3. Metodologia empregada





**2.1. CRIAÇÃO DE TABELA DE CONTROLE DE INGESTÃO**

In [0]:
%sql
CREATE OR REPLACE TABLE ControleIngestao
USING DELTA
AS

SELECT
    DATE_FORMAT(mes, 'yyyy-MM') AS periodo,
    mes AS data_inicio,
    ADD_MONTHS(mes, 1) AS data_fim,
    'PENDENTE' AS status,
    CAST(NULL AS BIGINT) AS quantidade_registros,
    CAST(NULL AS TIMESTAMP) AS data_ingestao
FROM (
    SELECT
        EXPLODE(
            SEQUENCE(
                DATE '2023-01-01',
                DATE '2026-06-01',
                INTERVAL 1 MONTH
            )
        ) AS mes
);

In [0]:
import requests
import time
from pyspark.sql import functions as fsql


#Variáveis utilizadas na chamada da API

url='https://api-publica.datajud.cnj.jus.br/api_publica_tjsp/_search'

headers = {
    "Authorization": "APIKey cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw==",
    "Content-Type": "application/json"
}


# Função de ingestão pela API
def extrair_processos(data_inicio, data_fim):

    consulta = {
        "size": 5000,
        "query": {
            "bool": {
                "filter": [
                    {
                        "term": {
                            "classe.codigo": 1116
                        }
                    },
                    {
                        "range": {
                            "dataAjuizamento": {
                                "gte": data_inicio,
                                "lt": data_fim
                            }
                        }
                    }
                ]
            }
        },
        "sort": [
            {"dataAjuizamento": "asc"}
        ]
    }

    Processos = []
    pagina = 1

    while True:

        tentativa = 1
        max_tentativas = 3

        while tentativa <= max_tentativas:

            response = requests.post(
                url,
                headers=headers,
                json=consulta
            )

            if response.status_code == 200:
                break

            if response.status_code in [429, 504]:

                print(
                    f"HTTP {response.status_code} - "
                    f"tentativa {tentativa}/{max_tentativas}"
                )

                time.sleep(10 * tentativa)
                tentativa += 1

            else:

                return (
                    False,
                    [],
                    f"Erro HTTP {response.status_code}"
                )

        # chegou ao limite de tentativas
        if response.status_code != 200:

            return (
                False,
                [],
                f"Erro HTTP {response.status_code} após {max_tentativas} tentativas"
            )

        dados = response.json()
        resultados = dados["hits"]["hits"]

        if len(resultados) == 0:
            break

        for processo in resultados:

            novo_processo = processo["_source"]

            novo_registro = {
                "numero_processo": novo_processo["numeroProcesso"],
                "tribunal": novo_processo["tribunal"],
                "data_ajuizamento": novo_processo["dataAjuizamento"],
                "grau": novo_processo["grau"],
                "classe_codigo": novo_processo["classe"]["codigo"],
                "classe_nome": novo_processo["classe"]["nome"],
                "orgaoJulgador_nome":
                    novo_processo["orgaoJulgador"]["nome"]
            }

            Processos.append(novo_registro)

        print(
            f"Página {pagina}: "
            f"{len(resultados)} registros"
        )

        pagina += 1

        ultimo_sort = resultados[-1]["sort"]
        consulta["search_after"] = ultimo_sort

        time.sleep(1.2)

    return True, Processos, None


# Ingestão de dados por período

periodos_pendentes = (
    spark.table("ControleIngestao")
    .filter(
        fsql.col("status").isin("PENDENTE", "ERRO")
    )
    .orderBy("data_inicio")
)

for linha in periodos_pendentes.collect():

    periodo = linha["periodo"]

    data_inicio = linha["data_inicio"].strftime("%Y%m%d%H%M%S")
    data_fim = linha["data_fim"].strftime("%Y%m%d%H%M%S")

    print(f"Iniciando período {periodo}")

    # Marca o período como PROCESSANDO
    spark.sql(f"""
        UPDATE ControleIngestao
        SET
            status = 'PROCESSANDO',
            mensagem_erro = NULL
        WHERE periodo = '{periodo}'
    """)

    sucesso, processos_mes, erro = extrair_processos(
        data_inicio,
        data_fim
    )

    if sucesso:

        if len(processos_mes) > 0:

            df_mes = spark.createDataFrame(processos_mes)

            df_mes = (
                df_mes
                .withColumn(
                    "periodo_ingestao",
                    fsql.lit(periodo)
                )
                .withColumn(
                    "data_ingestao",
                    fsql.current_timestamp()
                )
            )

            df_mes.write \
                .mode("append") \
                .saveAsTable("Bronze_Processos")

        # Atualiza controle após conclusão
        spark.sql(f"""
            UPDATE ControleIngestao
            SET
                status = 'OK',
                quantidade_registros = {len(processos_mes)},
                data_ingestao = CURRENT_TIMESTAMP(),
                mensagem_erro = NULL
            WHERE periodo = '{periodo}'
        """)

        print(
            f"{periodo} concluído: "
            f"{len(processos_mes)} registros."
        )

    else:

        spark.sql(f"""
            UPDATE ControleIngestao
            SET
                status = 'ERRO',
                quantidade_registros = NULL,
                mensagem_erro = '{erro}'
            WHERE periodo = '{periodo}'
        """)

        print(
            f"Falha na ingestão de {periodo}: {erro}"
        )


In [0]:
%sql
SELECT periodo_ingestao, COUNT(numero_processo)
FROM Bronze_Processos
GROUP BY periodo_ingestao
ORDER BY periodo_ingestao;

In [0]:
%sql
SELECT *
FROM Bronze_Processos
LIMIT 1;

In [0]:
%sql
CREATE OR REPLACE TABLE Silver_Processos AS

WITH tratamento AS (
    SELECT
        numero_processo,
        tribunal,
        COALESCE(TRY_TO_TIMESTAMP(data_ajuizamento, 'yyyyMMddHHmmss'), TRY_CAST(data_ajuizamento AS TIMESTAMP)) AS data_ajuizamento,
        grau,
        classe_codigo,
        classe_nome,
        orgaoJulgador_nome
    FROM Bronze_Processos)

SELECT
    numero_processo,
    tribunal,
    TO_DATE(data_ajuizamento) as data_ajuizamento,
    YEAR(data_ajuizamento) AS ano,
    MONTH(data_ajuizamento) AS mes,
    DATE_FORMAT(data_ajuizamento, 'yyyy-MM') AS ano_mes_ajuizamento,
    grau,
    orgaoJulgador_nome
FROM tratamento;


In [0]:
%sql
SELECT DISTINCT COUNT(numero_processo)
FROM Silver_Processos;

Concluída a etapa de transformação, com a consolidação dos resultados no nível Silver, passamos à etapa referente ao **nível Ouro** do pipeline de dados.

 Para resposta ao questionamento inicial, faz-se necessário quantificar o comportamento da quantidade de processos de execução fiscal ajuizados ao longo do período de apuração deste trabalho, bem como a sua distribuição entre as unidades do Tribunal de Justiça do Estado de São Paulo.

Logo, a entidade central do modelo de dados seriam os processos, os quais têm, entre seus atributos, a data de ajuizamento, que corresponde à data em que ocorreu o seu protocolo. Por outro lado, essa entidade possui relacionamento com a entidade "Órgão Julgador", que representa a unidade judiciária competente para o processamento daquele processo.

A partir desse arranjo, teremos uma **modelagem em estrela** com os seguintes componentes:

1) Tabela-Fato: **Processos**
2) Tabela-Dimensão: **Tempo**
3) Tabela-Dimensão: **ÓrgãoJulgador**

In [0]:
%sql
CREATE OR REPLACE TABLE Gold_OrgaoJulgador AS

SELECT 
    ROW_NUMBER() OVER (ORDER BY orgaoJulgador_nome) AS id_orgao,
    orgaojulgador_nome as nome
FROM (
    SELECT DISTINCT orgaoJulgador_nome
    FROM Silver_processos
);


SELECT COUNT (DISTINCT id_orgao)
FROM Gold_OrgaoJulgador





In [0]:
%sql
CREATE OR REPLACE TABLE Gold_Tempo AS

SELECT DISTINCT
CAST(DATE_FORMAT(TO_DATE(data_ajuizamento), 'yyyyMMdd') AS INT) AS id_data,
ano_mes_ajuizamento,
ano,
mes,
TO_DATE(data_ajuizamento) AS data_ajuizamento
FROM Silver_processos;


In [0]:
%sql
SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT id_data) AS ids_distintos,
    COUNT(DISTINCT data_ajuizamento) AS datas_distintas,
    MIN(data_ajuizamento) AS menor_data,
    MAX(data_ajuizamento) AS maior_data
FROM Gold_Tempo;

In [0]:
%sql
CREATE OR REPLACE TABLE Gold_FatoAjuizamento AS

SELECT
    p.numero_processo,
    t.id_data,
    o.id_orgao,
    p.grau
FROM Silver_processos p

LEFT JOIN Gold_Tempo t ON p.data_ajuizamento = t.data_ajuizamento

LEFT JOIN Gold_OrgaoJulgador o ON p.orgaoJulgador_nome = o.nome;



In [0]:
%sql
SELECT COUNT (*)
FROM Gold_FatoAjuizamento;

In [0]:
%sql
SELECT gold_Tempo.ano_mes_ajuizamento, COUNT (numero_processo)
FROM gold_fatoajuizamento
JOIN gold_tempo on gold_fatoajuizamento.id_data = Gold_tempo.id_data
GROUP BY Gold_Tempo.ano_mes_ajuizamento
ORDER BY ano_mes_ajuizamento;
    


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    gold_orgaojulgador.nome, 
    count(numero_processo) As Total_Execucoes
FROM gold_fatoajuizamento
JOIN gold_orgaojulgador on gold_fatoajuizamento.id_orgao = gold_orgaojulgador.id_orgao
GROUP BY gold_orgaojulgador.nome
ORDER BY total_execucoes DESC;